In [0]:
%pip install -qqqq -U mlflow[genai,databricks] databricks-sdk databricks-openai
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("genie_agent_id", "", "Genie Agent ID")
dbutils.widgets.text("experiment_name", "", "Experiment Name")
AGENT_ID = dbutils.widgets.get("genie_agent_id")
EXPERIMENT_NAME = dbutils.widgets.get("experiment_name")

## [Step 1: Configure](https://mlflow.org/cookbook/genie-space-analyzer/#step-1-configure)

In [0]:
from databricks.sdk import WorkspaceClient
from databricks_openai import DatabricksOpenAI
import json
import mlflow

w = WorkspaceClient()

# Create an OpenAI client that is connected to Databricks-hosted LLMs
client = DatabricksOpenAI()

mlflow.set_experiment(EXPERIMENT_NAME)

In [0]:
import os

default_warehouse = next(
    (
        wh
        for wh in w.warehouses.list()
        if "Serverless Starter Warehouse" in wh.name and wh.enable_serverless_compute
    ),
    None,
)
default_warehouse_id = default_warehouse.id if default_warehouse else None
print(f"{default_warehouse_id=}")

# Specify the ID of a SQL warehouse you have access to.
os.environ["MLFLOW_TRACING_SQL_WAREHOUSE_ID"] = default_warehouse_id

## [Step 2: Load Failed Traces from Evaluation](https://mlflow.org/cookbook/genie-space-analyzer/#step-2-load-failed-traces-from-evaluation)

In [0]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
all_traces = mlflow.search_traces(
    locations=[experiment.experiment_id],
    return_type="list",
)

failed_conversations = []
for trace in all_traces:
    assessments = trace.info.assessments or []
    failures = [a for a in assessments if a.value == "no"]
    if not failures:
        continue

    root = trace.data.spans[0]
    failed_conversations.append(
        {
            "question": json.loads(root.inputs).get("question"),
            "response": json.loads(root.outputs).get("response"),
            "generated_sql": json.loads(root.outputs).get("generated_sql"),
            "error": json.loads(root.outputs).get("error"),
            "failed_checks": [f"{a.name}: {a.value} - {a.rationale}" for a in failures],
        }
    )

print(f"{len(failed_conversations)} / {len(all_traces)} " f"traces had failures")

## [Step 3: Extract the Genie Space Configuration](https://mlflow.org/cookbook/genie-space-analyzer/#step-3-extract-the-genie-space-configuration)

In [0]:
space = w.genie.get_space(space_id=AGENT_ID, include_serialized_space=True)
config = json.loads(space.serialized_space) if space.serialized_space else {}

tables = config.get("data_sources", {}).get("tables", [])
instructions = config.get("instructions", {})
text_instructions = instructions.get("text_instructions", [])
example_sqls = instructions.get("example_question_sqls", [])

print(f"Space: {space.title}")
print(f"Tables: {len(tables)}, " f"Instructions: {len(text_instructions)}")

## [Step 4: Generate Fixes with an LLM](https://mlflow.org/cookbook/genie-space-analyzer/#step-4-generate-fixes-with-an-llm)

In [0]:
table_names = [t["identifier"] for t in tables]

system_prompt = (
    "You are an expert Databricks AI/BI Genie space consultant. "
    "You will be given conversations where Genie gave wrong or "
    "incomplete answers, along with the specific checks that failed. "
    "Generate specific, copy-paste-ready fixes: SQL expressions, "
    "text instructions, example SQL, and column descriptions. "
    "Never give vague advice. Always write the actual implementation."
)

analysis_prompt = f"""Fix the issues found in these Genie conversations.

## FAILED CONVERSATIONS
{json.dumps(failed_conversations[:20], indent=2)}

## CURRENT SPACE CONFIG
Title: {space.title}
Tables: {', '.join(table_names[:10])}
Text instructions: {len(text_instructions)}
Example SQL: {len(example_sqls)}

For each failed conversation, provide a specific fix: a new text
instruction, SQL expression, example query, or column description
that would prevent the failure. Prioritize by impact."""


@mlflow.trace
def analyze_genie_space(user_prompt, sys_prompt):
    response = client.chat.completions.create(
        model="databricks-claude-sonnet-4-6",
        messages=[
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": user_prompt},
        ],
        max_tokens=8000,
        temperature=0.1,
    )
    return response.choices[0].message.content


if not failed_conversations:
    print("No failures found - nothing to analyze!")
else:
    recommendations = analyze_genie_space(analysis_prompt, system_prompt)
    print(recommendations)